# VigIA — treino do modelo no Colab

In [ ]:
!pip install -q ultralytics
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA - troque o ambiente para GPU')

## Enviar o `vigia_dataset.zip` com as fotos mapeadas

In [ ]:
from google.colab import files
enviado = files.upload()  # escolha vigia_dataset.zip
!rm -rf /content/yolo && mkdir -p /content/yolo
!unzip -q -o vigia_dataset.zip -d /content/yolo
!ls /content/yolo; echo; echo treino: $(ls /content/yolo/images/train | wc -l) imagens; echo val: $(ls /content/yolo/images/val | wc -l) imagens

## Treinar

In [ ]:
from pathlib import Path
Path('/content/yolo/vigia.yaml').write_text('''path: /content/yolo
train: images/train
val: images/val
kpt_shape: [2, 3]
flip_idx: [0, 1]
names:
  0: carro
''')

from ultralytics import YOLO
modelo = YOLO('yolo11n-pose.pt')
modelo.train(data='/content/yolo/vigia.yaml', epochs=100, imgsz=640, batch=16,
             project='/content/runs', name='vigia', exist_ok=True, patience=30,
             degrees=180.0, flipud=0.5, fliplr=0.5)
print('melhor modelo:', modelo.trainer.best)

## Validação por imagens

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob
melhor = YOLO('/content/runs/vigia/weights/best.pt')
for r in melhor.predict(sorted(glob.glob('/content/yolo/images/val/*.jpg'))[:6], conf=0.5, save=True,
                        project='/content/runs', name='amostras', exist_ok=True, verbose=False):
    pass
for f in sorted(glob.glob('/content/runs/amostras/*.jpg'))[:6]:
    display(Image(filename=f, width=480))
display(Image(filename='/content/runs/vigia/results.png', width=900))

## Baixar o modelo
Salve como `modelos/vigia-pose.pt` dentro da pasta do projeto.

In [ ]:
import shutil
shutil.copy('/content/runs/vigia/weights/best.pt', '/content/vigia-pose.pt')
files.download('/content/vigia-pose.pt')